# Inhoudsopgave
## Documentatie / uitleg
- Productoverzicht
- Stakeholder analyse
- Datavereisten
- Modelvereisten
- Onderhoud en hertraining
- Data pipeline
- Modellering
- Deployment
- CI/CD
- Monitoring
## Technisch onderdeel
- Data loading
- Data preprocessing and feature engineering
- Model training
- Deployment
- Monitoring

# Documentatie & uitleg

Hieronder wordt uitleg gegeven over de inhoud van het notebook.
### Productoverzicht:
Dit project maakt een intelligent retail analytics systeem voor BrightMart, een middelgrote retailer. Het doel van het systeem is om winkels efficiënter te maken door edge-based klantdetectie te combineren met cloudgebaseerde vraagvoorspellingen.

Het systeem bestaat uit twee modellen:
- Een cloud based model dat de vraag naar producten voorspelt op basis van historische verkoopgegevens.
- Een edge model dat het aantal klanten in een winkel inschat op basis van camerabeelden

Met dit systeem kunnen er real-time inzichten verkregen worden voor store managers en kan de voorraad beter bijgehouden worden voor het supply chain team.

### Stakeholders:
- Store managers: Hebben behoefte aan real time inzicht van de winkel bezetting en de omzet van een winkel.
- Supply chain team: Hebben behoefte aan een nauwkeurige voorspelling van verkoop.
- IT-afdeling: Wilt een data pipeline waar niet veel aan gedaan hoeft te worden.

### Datavereisten:
Het systeem maakt gebruik van twee soorten data:
- Retaildata: Deze data bevat informatie over winkel, product, datum en verkoop.
- Beelddata: Deze wordt gebruikt om klanten te detecteren en te tellen.

De eisen van de data zijn als volgt:
- Kwaliteit: De data moet schoon zijn. (Missende waardes worden verwijderd)
- Volume: De pipeline is schaalbaar en kan volume aan.
- Snelheid: Het edge model moet real-time voorspellingen kunnen maken.
- Privacy: De beelddata wordt lokaal verwerkt en niet opgeslagen.
- Veiligheid: Data wordt veilig opgeslagen en verwerkt binnen een beveiligd platform
- Vorm: De beelddata wordt in .npy bestanden aangeleverd.

### Model vereisten
Er worden twee modellen gebruikt:

#### Cloud model:
- Taak: Voorspellen van het aan verkochte items van aankomende dagen zodat het supply chain team spullen kan inkopen.
- Type model: RandomForestRegressor (SparkML)
- Eisen:
  - Zo laag mogelijke RMSE
  - Schaalbaar
  - MLflow integratie voor tracking en versiebeheer

#### Edge model:
- Taak: Het aantal klanten in een winkel voorspellen aan de hand van camerabeelden.
- Type model: Regressie (Lineaire regressie of RandomForest)
- Eisen:
  - Real time voorspellingen kunnen maken
  - Zo laag mogelijke RMSE

#### Onderhoud & hertraining:
Het systeem wordt ontworpen om zich aan te passen aan verandering.
- Modelprestaties worden gemonitord aan de hand van de RMSE.
- Data drift wordt gedetecteerd, bij data drift krijgt gebruiker een melding om hertraining in te plannen.
- Data drift detectie kan per model worden aangepast naar andere hoeveelheid.

#### Data pipeline:
De data pipeline bestaat uit:
- Data ingestion: Het inladen van de retail- en beelddata.
- Data cleaning: Het verwijderen van ongeldige en dubbele waardes
- Feature engineering: Het toevoegen van features zoals datum componenten en lag-variabelen.
- Data splitsen: Het maken van train/test sets.

#### Modellering:
De modelleringspipeline bevat:
- Feature engineering (VectorAssembler voor SparkML)
- Model training (RandomForest en Lineaire regressie)
- Evaluatie met RMSE
- Experiment tracking met behulp van MLFlow

#### Deployment:
De modellen worden geladen vanuit MLFlow en gebruikt voor voorspellingen.

#### CI/CD:
Het systeem ondersteund het volgende:
- Continuous Integration: Modellen worden bijgehouden in MLFlow.
- Continuous Deployment: Nieuwe modellen kunnen toegepast worden.

#### Monitoring:
Modelprestaties worden gemonitord:
- RMSE voor evaluatie.
- Drift detection vergelijkt voorspellingen met echte waardes.
- Bij afwijkingen krijgt gebruiker een melding om modellen opnieuw te trainen.

# Technisch onderdeel

In [0]:
from pyspark.sql.types import *
from pyspark.sql.functions import *
from pyspark.ml import Pipeline
from pyspark.ml.feature import VectorAssembler
from pyspark.ml.regression import RandomForestRegressor
from pyspark.ml.evaluation import RegressionEvaluator
from pyspark.sql.window import Window


from sklearn.metrics import mean_squared_error
from sklearn.ensemble import RandomForestRegressor

from PIL import Image
import mlflow
import os
import numpy as np
import pandas as pd

# Data loading

In [0]:

store_demand_schema = StructType([
  StructField('date', DateType(), False),
  StructField('store', IntegerType(), False),
  StructField('item', IntegerType(), False),
  StructField('sales', IntegerType(), False)
])
ecom_schema = StructType([
  StructField('InvoiceNo', IntegerType(), False),
  StructField('StockCode', IntegerType(), False),
  StructField('Description', StringType(), False),
  StructField('Quantity', IntegerType(), False),
  StructField('InvoiceDate', StringType(), False),
  StructField('UnitPrice', FloatType(), False),
  StructField('CustomerID', IntegerType(), False),
  StructField('Country', StringType(), False)
])

base_path = "/Volumes/workspace/default/course_files/"

ecom_df = spark.read.csv(
    base_path + "ecom/data.csv",
    header=True,
    schema=ecom_schema
)

retail_train = spark.read.csv(
    base_path + "retail/train.csv",
    header=True,
    schema=store_demand_schema
)

retail_test = spark.read.csv(
    base_path + "retail/test.csv",
    header=True,
    schema=store_demand_schema
)

images_df = spark.read.format("binaryFile").load(
    base_path + "person_detection_small/images/"
).select("path")

images = np.load(base_path + "surv_camera/images_400.npy")
labels = np.load(base_path + "surv_camera/labels_400.npy")

print("Ecom rows:", ecom_df.count())
print("Retail train rows:", retail_train.count())

print("Images numpy shape:", images.shape)
print("Labels numpy shape:", labels.shape)

display(ecom_df)
display(retail_train)
display(images_df.limit(5))

Ecom rows: 541909
Retail train rows: 913000
Images numpy shape: (400, 480, 640, 3)
Labels numpy shape: (400, 1)


InvoiceNo,StockCode,Description,Quantity,InvoiceDate,UnitPrice,CustomerID,Country
536365,null,WHITE HANGING HEART T-LIGHT HOLDER,6,12/1/2010 8:26,2.55,17850,United Kingdom
536365,71053,WHITE METAL LANTERN,6,12/1/2010 8:26,3.39,17850,United Kingdom
536365,null,CREAM CUPID HEARTS COAT HANGER,8,12/1/2010 8:26,2.75,17850,United Kingdom
536365,null,KNITTED UNION FLAG HOT WATER BOTTLE,6,12/1/2010 8:26,3.39,17850,United Kingdom
536365,null,RED WOOLLY HOTTIE WHITE HEART.,6,12/1/2010 8:26,3.39,17850,United Kingdom
536365,22752,SET 7 BABUSHKA NESTING BOXES,2,12/1/2010 8:26,7.65,17850,United Kingdom
536365,21730,GLASS STAR FROSTED T-LIGHT HOLDER,6,12/1/2010 8:26,4.25,17850,United Kingdom
536366,22633,HAND WARMER UNION JACK,6,12/1/2010 8:28,1.85,17850,United Kingdom
536366,22632,HAND WARMER RED POLKA DOT,6,12/1/2010 8:28,1.85,17850,United Kingdom
536367,84879,ASSORTED COLOUR BIRD ORNAMENT,32,12/1/2010 8:34,1.69,13047,United Kingdom


date,store,item,sales
2013-01-01,1,1,13
2013-01-02,1,1,11
2013-01-03,1,1,14
2013-01-04,1,1,13
2013-01-05,1,1,10
2013-01-06,1,1,12
2013-01-07,1,1,10
2013-01-08,1,1,9
2013-01-09,1,1,12
2013-01-10,1,1,9


path
dbfs:/Volumes/workspace/default/course_files/person_detection_small/images/09ec32608cf39bf9.jpg
dbfs:/Volumes/workspace/default/course_files/person_detection_small/images/0b78c7d88fbec271.jpg
dbfs:/Volumes/workspace/default/course_files/person_detection_small/images/08f2f2fd37ba8f90.jpg
dbfs:/Volumes/workspace/default/course_files/person_detection_small/images/0822e260ff59a134.jpg
dbfs:/Volumes/workspace/default/course_files/person_detection_small/images/02c20a4149814628.jpg


# Data preprocessing and feature engineering

Hieronder worden functies aangemaakt om de data op te schonen en extra features aan te maken. Dit wordt gedaan middels zelf-gedefineerde functies zodat er modulair gewerkt kan worden en het makkelijk is om dingen aan te passen of te veranderen. 

Daarna worden de dataframes bewerkt door middel van de functies en wordt het dataframe klaargemaakt om gebruikt te worden in een machine-learning model.

In [0]:
def clean_ecom(df):
    return df \
        .filter(col("Quantity") > 0) \
        .filter(col("StockCode").isNotNull()) \
        .dropDuplicates()

def clean_retail(df):
    return df \
        .filter(col("sales") > 0) \
        .dropDuplicates() \
        .dropna()

def create_retail_features(df):
    return df \
        .withColumn("year", year("date")) \
        .withColumn("month", month("date")) \
        .withColumn("day", dayofmonth("date"))

def create_ecom_features(df):
    return df \
        .withColumn("revenue", col("Quantity") * col("UnitPrice")) \
        .withColumn("year", year("InvoiceDate")) \
        .withColumn("month", month("InvoiceDate")) \
        .withColumn("day", dayofmonth("InvoiceDate")) \
        .withColumn("day_of_week", dayofweek("InvoiceDate"))


def create_retail_gold(df):
    return df.groupBy("store", "item", "year", "month", "day").agg(
    sum("sales").alias("sales")
    )

def validate_df(df):
    print("Rows:", df.count())
    df.printSchema()

In [0]:
ecom_clean = clean_ecom(ecom_df)
retail_clean = clean_retail(retail_train)

retail_features = create_retail_features(retail_clean)
ecom_features = create_ecom_features(ecom_clean)

retail_gold = create_retail_gold(retail_features)

# Lags toevoegen om beter te kunnen voorspellen.
window = Window.partitionBy("store", "item").orderBy("year", "month", "day") #Window maken waarover lags worden gemaakt
retail_gold = retail_gold \
    .withColumn("lag_1", lag("sales", 1).over(window)) \
    .withColumn("lag_7", lag("sales", 7).over(window)) #Lag 1 en 7 aanmaken en toevoegen aan dataframe
retail_gold = retail_gold.dropna() # NaN's verwijderen omdat die moet bij lags, dat 1 heeft namelijk geen lag.

validate_df(ecom_df)
validate_df(retail_train)
validate_df(ecom_clean)
validate_df(retail_clean)
validate_df(retail_features)
validate_df(ecom_features)
validate_df(retail_gold)


Rows: 541909
root
 |-- InvoiceNo: integer (nullable = true)
 |-- StockCode: integer (nullable = true)
 |-- Description: string (nullable = true)
 |-- Quantity: integer (nullable = true)
 |-- InvoiceDate: string (nullable = true)
 |-- UnitPrice: float (nullable = true)
 |-- CustomerID: integer (nullable = true)
 |-- Country: string (nullable = true)

Rows: 913000
root
 |-- date: date (nullable = true)
 |-- store: integer (nullable = true)
 |-- item: integer (nullable = true)
 |-- sales: integer (nullable = true)

Rows: 473201
root
 |-- InvoiceNo: integer (nullable = true)
 |-- StockCode: integer (nullable = true)
 |-- Description: string (nullable = true)
 |-- Quantity: integer (nullable = true)
 |-- InvoiceDate: string (nullable = true)
 |-- UnitPrice: float (nullable = true)
 |-- CustomerID: integer (nullable = true)
 |-- Country: string (nullable = true)

Rows: 912999
root
 |-- date: date (nullable = true)
 |-- store: integer (nullable = true)
 |-- item: integer (nullable = true)
 |-

# Model training

## Cloud model training
Hieronder wordt er een cloud model gemaakt om een voorspelling te maken van het aantal "items" wat verkocht gaat worden. Dit wordt aan de hand van een RandomForestRegressor model gedaan omdat deze een aantal voordelen heeft:
  - Robuust tegen ruis
  - Kan niet lineaire verbanden ontdekken
  - Heeft geen ingewikkelde pre-processing nodig\
Hierdoor is RandomForestRegressor een goed baseline model.

Er wordt een assembler gebruikt die alle features omzet naar één vector omdat spark modellen zo werken.

In [0]:
# Assembler aanmaken
assembler = VectorAssembler(
    inputCols=["store", "item", "year", "month", "day", "lag_1", "lag_7"],
    outputCol="features"
)

# RandomForest model aanmaken
rf = RandomForestRegressor(
    featuresCol="features",
    labelCol="sales"
)
# Pipeline bouwen
pipeline = Pipeline(stages=[assembler, rf])

train, test = retail_gold.randomSplit([0.8, 0.2], seed=42)

In [0]:
# Evaluator aanmaken met RMSE als metric
evaluator = RegressionEvaluator(
    labelCol="sales",
    predictionCol="prediction",
    metricName="rmse"
)

# (Tijdelijke) Opslaglocatie voor modellen aanmaken
os.environ["MLFLOW_DFS_TMP"] = "/Volumes/workspace/default/course_files/tmp/"

# Met een kleine grid RandomForest fitten
depths = [5, 10, 15]
for depth in depths:
    with mlflow.start_run():
        
        rf = RandomForestRegressor(
            maxDepth=depth,
            featuresCol="features",
            labelCol="sales"
        )
        
        pipeline = Pipeline(stages=[assembler, rf])
        
        model = pipeline.fit(train)
        preds = model.transform(test)
        
        rmse = evaluator.evaluate(preds)
        
        # Logging
        mlflow.log_param("maxDepth", depth)
        mlflow.log_metric("rmse", rmse)
        mlflow.spark.log_model(model, f"model_depth_{depth}")
        
        print(f"Depth {depth} → RMSE: {rmse}")

2026/04/29 10:56:18 WARNING mlflow.utils.requirements_utils: Found pyspark version (4.0.0+databricks.connect.17.3.7) contains a local version label (+databricks.connect.17.3.7). MLflow logged a pip requirement for this package as 'pyspark==4.0.0' without the local version label to make it installable from PyPI. To specify pip requirements containing local version labels, please use `conda_env` or `pip_requirements`.
2026/04/29 10:56:21 WARNING mlflow.utils.environment: Encountered an unexpected error while inferring pip requirements (model URI: /local_disk0/user_tmp_data/spark-06958379-bf99-4ddc-8f47-85/tmpyitui5y5/model, flavor: spark). Fall back to return ['pyspark==4.0.0']. Set logging level to DEBUG to see the full traceback. 
2026/04/29 10:56:21 WARNING mlflow.models.model: Model logged without a signature and input example. Please set `input_example` parameter when logging the model to auto infer the model signature.


Depth 5 → RMSE: 9.962707415865857


2026/04/29 10:56:50 WARNING mlflow.utils.requirements_utils: Found pyspark version (4.0.0+databricks.connect.17.3.7) contains a local version label (+databricks.connect.17.3.7). MLflow logged a pip requirement for this package as 'pyspark==4.0.0' without the local version label to make it installable from PyPI. To specify pip requirements containing local version labels, please use `conda_env` or `pip_requirements`.
2026/04/29 10:56:52 WARNING mlflow.utils.environment: Encountered an unexpected error while inferring pip requirements (model URI: /local_disk0/user_tmp_data/spark-06958379-bf99-4ddc-8f47-85/tmp5bejjxid/model, flavor: spark). Fall back to return ['pyspark==4.0.0']. Set logging level to DEBUG to see the full traceback. 
2026/04/29 10:56:52 WARNING mlflow.models.model: Model logged without a signature and input example. Please set `input_example` parameter when logging the model to auto infer the model signature.


Depth 10 → RMSE: 9.452819719445609


2026/04/29 10:58:23 WARNING mlflow.utils.requirements_utils: Found pyspark version (4.0.0+databricks.connect.17.3.7) contains a local version label (+databricks.connect.17.3.7). MLflow logged a pip requirement for this package as 'pyspark==4.0.0' without the local version label to make it installable from PyPI. To specify pip requirements containing local version labels, please use `conda_env` or `pip_requirements`.
2026/04/29 10:58:25 WARNING mlflow.utils.environment: Encountered an unexpected error while inferring pip requirements (model URI: /local_disk0/user_tmp_data/spark-06958379-bf99-4ddc-8f47-85/tmph4epf92t/model, flavor: spark). Fall back to return ['pyspark==4.0.0']. Set logging level to DEBUG to see the full traceback. 
2026/04/29 10:58:25 WARNING mlflow.models.model: Model logged without a signature and input example. Please set `input_example` parameter when logging the model to auto infer the model signature.


Depth 15 → RMSE: 9.31612199916215


### Experiment log voor cloud model
Hieronder worden de resultaten van de verschillende modellen getoond, er is duidelijk te zien dat de lag functies de prestaties hebben verbeterd. Index 4-5-6 zijn de modellen die getraind zijn zonder lag features en index 1-2-3 zijn de modellen die wel zijn getraind met lag features. Binnen het nieuwe model maakt het minder uit hoe uitgebreid de maxDepth is, daarom kan er een afweging gemaakt worden over wat belangrijker is. Snelheid van trainen, accuraatheid of generalisatie. Hier wordt later meer over verteld.

In [0]:
log = mlflow.search_runs()
log_performance = log[["run_id", "experiment_id","metrics.rmse","params.maxDepth"]]
display(log_performance[2:])
display(log[2:])

run_id,experiment_id,metrics.rmse,params.maxDepth
3ae4fd603d0448f09cb4d649381e6664,541341117399548,9.31612199916215,15
1de753c01b214a37b21668b9a76bdabe,541341117399548,9.452819719445609,10
ce70553bf9494add844e4050b417480a,541341117399548,9.962707415865857,5
19f200980c234113a14b4b7626bd46eb,541341117399548,18.721329528076506,15
5904828da9834b978fd05d6377a00539,541341117399548,20.976303584936865,10
e4ffde6f6d7741ffa902cbf1afcc7118,541341117399548,24.06325643434852,5


run_id,experiment_id,status,artifact_uri,start_time,end_time,metrics.rmse,params.model_type,params.n_estimators,params.maxDepth,tags.mlflow.databricks.cluster.info,tags.mlflow.source.name,tags.mlflow.user,tags.mlflow.runName,tags.mlflow.runColor,tags.mlflow.databricks.notebook.commandID,tags.mlflow.databricks.workspaceURL,tags.mlflow.databricks.notebookRevisionID,tags.mlflow.log-model.history,tags.mlflow.databricks.cluster.libraries,tags.mlflow.databricks.cluster.id,tags.mlflow.databricks.notebookID,tags.mlflow.databricks.notebookPath,tags.mlflow.databricks.workspaceID,tags.mlflow.databricks.webappURL,tags.mlflow.source.type
3ae4fd603d0448f09cb4d649381e6664,541341117399548,FINISHED,dbfs:/databricks/mlflow-tracking/541341117399548/3ae4fd603d0448f09cb4d649381e6664/artifacts,2026-04-29T10:56:53.788Z,2026-04-29T10:58:27.148Z,9.31612199916215,null,null,15,"{""cluster_name"":"""",""spark_version"":""client.4.10-aarch64-scala2.13"",""autotermination_minutes"":120}",/MLOPSPortfolio/notebooks/MLOPSPortfolio,ryan.huismans99@gmail.com,welcoming-gull-102,#5bc5db,1777457387725_8718615021316209485_a9c4ec99d7c346d6b777cc231a71a73a,https://dbc-f91f8233-2c2e.cloud.databricks.com,1777460307249,"[{""artifact_path"":""model_depth_15"",""flavors"":{""spark"":{""pyspark_version"":""4.0.0"",""model_data"":""sparkml"",""code"":null,""model_class"":""pyspark.ml.pipeline.PipelineModel""},""python_function"":{""loader_module"":""mlflow.spark"",""python_version"":""3.12.3"",""data"":""sparkml"",""env"":{""conda"":""conda.yaml"",""virtualenv"":""python_env.yaml""}}},""utc_time_created"":""2026-04-29 10:58:09.572720""}]","{""installable"":[],""redacted"":[]}",0429-101358-wrutjzbu-v2n,541341117399548,/MLOPSPortfolio/notebooks/MLOPSPortfolio,7474647629856771,https://dbc-f91f8233-2c2e.cloud.databricks.com,NOTEBOOK
1de753c01b214a37b21668b9a76bdabe,541341117399548,FINISHED,dbfs:/databricks/mlflow-tracking/541341117399548/1de753c01b214a37b21668b9a76bdabe/artifacts,2026-04-29T10:56:22.982Z,2026-04-29T10:56:53.681Z,9.452819719445609,null,null,10,"{""cluster_name"":"""",""spark_version"":""client.4.10-aarch64-scala2.13"",""autotermination_minutes"":120}",/MLOPSPortfolio/notebooks/MLOPSPortfolio,ryan.huismans99@gmail.com,blushing-mule-379,#edb732,1777457387725_8718615021316209485_a9c4ec99d7c346d6b777cc231a71a73a,https://dbc-f91f8233-2c2e.cloud.databricks.com,1777460213758,"[{""artifact_path"":""model_depth_10"",""flavors"":{""spark"":{""pyspark_version"":""4.0.0"",""model_data"":""sparkml"",""code"":null,""model_class"":""pyspark.ml.pipeline.PipelineModel""},""python_function"":{""loader_module"":""mlflow.spark"",""python_version"":""3.12.3"",""data"":""sparkml"",""env"":{""conda"":""conda.yaml"",""virtualenv"":""python_env.yaml""}}},""utc_time_created"":""2026-04-29 10:56:41.190344""}]","{""installable"":[],""redacted"":[]}",0429-101358-wrutjzbu-v2n,541341117399548,/MLOPSPortfolio/notebooks/MLOPSPortfolio,7474647629856771,https://dbc-f91f8233-2c2e.cloud.databricks.com,NOTEBOOK
ce70553bf9494add844e4050b417480a,541341117399548,FINISHED,dbfs:/databricks/mlflow-tracking/541341117399548/ce70553bf9494add844e4050b417480a/artifacts,2026-04-29T10:55:59.882Z,2026-04-29T10:56:22.872Z,9.962707415865857,null,null,5,"{""cluster_name"":"""",""spark_version"":""client.4.10-aarch64-scala2.13"",""autotermination_minutes"":120}",/MLOPSPortfolio/notebooks/MLOPSPortfolio,ryan.huismans99@gmail.com,redolent-swan-935,#c565c7,1777457387725_8718615021316209485_a9c4ec99d7c346d6b777cc231a71a73a,https://dbc-f91f8233-2c2e.cloud.databricks.com,1777460183014,"[{""artifact_path"":""model_depth_5"",""flavors"":{""spark"":{""pyspark_version"":""4.0.0"",""model_data"":""sparkml"",""code"":null,""model_class"":""pyspark.ml.pipeline.PipelineModel""},""python_function"":{""loader_module"":""mlflow.spark"",""python_version"":""3.12.3"",""data"":""sparkml"",""env"":{""conda"":""conda.yaml"",""virtualenv"":""python_env.yaml""}}},""utc_time_created"":""2026-04-29 10:56:09.081291""}]","{""installable"":[],""

## Edge model training
Nu er een cloudmodel is getraind om de verkoop te voorspellen, wordt een edge model ontwikkeld dat het aantal klanten in de winkel inschat. Dit model maakt gebruik van camerabeelden en wordt lokaal uitgevoerd op edge-apparaten voor real-time inzichten.

Voor het edge model worden twee regressiemodellen vergeleken: lineaire regressie en een Random Forest model. Aangezien de dataset beschikbaar is als NumPy-arrays (.npy), worden de beelden direct als numerieke input gebruikt, zonder een volledige computer vision pipeline te implementeren.

In plaats van complexe deep learning-modellen is bewust gekozen voor lichte modellen. Deze keuze is gemaakt om te voldoen aan de beperkingen van edge-apparaten, zoals beperkte rekenkracht en geheugen.

Om beide modellen te trainen worden er twee datasets gemaakt, een kleine variant waar de images verkleind worden omdat RandomForest er te lang over doet.  

In [0]:
# Images dataset verkleinen naar 64x64
images_small = np.array([
    np.array(Image.fromarray(img).resize((64, 64)))
    for img in images
])
# Labels defineren (aantal mensen in de foto)
y = labels

# Images hervormen zodat ze als input gebruikt kunnen worden. 
X_small = images_small.reshape(len(images_small), -1)

# Train test split maken
X_train_small, X_test_small, y_train_small, y_test_small = train_test_split(
    X_small, y, test_size=0.2, random_state=42
)

# Images hervormen zodat ze als input gebruikt kunnen worden
X = images.reshape(len(images), -1)

# Train test split maken
X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, random_state=42
)

In [0]:
# (Tijdelijke) opslagplek voor de modellen defineren
os.environ["MLFLOW_DFS_TMP"] = "/Volumes/workspace/default/course_files/tmp/"

# Lineaire regressie
with mlflow.start_run(run_name="LinearRegression"):
    
    model = LinearRegression()
    model.fit(X_train, y_train)
    
    preds = model.predict(X_test)
    mse = mean_squared_error(y_test, preds)
    rmse = np.sqrt(mse)
    
    mlflow.log_param("model_type", "LinearRegression")
    mlflow.log_metric("rmse", rmse)
    
    mlflow.sklearn.log_model(model, "edge_model_linear")
    
    print("Linear RMSE:", rmse)

# RandomForestRegressor
with mlflow.start_run(run_name="RandomForest"):
    
    model = RandomForestRegressor(n_estimators=50)
    model.fit(X_train_small, y_train_small)
    
    preds = model.predict(X_test_small)
    mse = mean_squared_error(y_test_small, preds)
    rmse = np.sqrt(mse)
    
    mlflow.log_param("model_type", "RandomForest")
    mlflow.log_param("n_estimators", 50)
    mlflow.log_metric("rmse", rmse)
    
    mlflow.sklearn.log_model(model, "edge_model_rf")
    
    print("RF RMSE:", rmse)

2026/04/29 12:17:48 WARNING mlflow.models.model: Model logged without a signature and input example. Please set `input_example` parameter when logging the model to auto infer the model signature.


Linear RMSE: 3.2409702058105787


/databricks/python_shell/lib/dbruntime/MLWorkloadsInstrumentation/_sklearn.py:29: DataConversionWarning: A column-vector y was passed when a 1d array was expected. Please change the shape of y to (n_samples,), for example using ravel().
  original_result = original(self, *args, **kwargs)
2026/04/29 12:19:00 WARNING mlflow.models.model: Model logged without a signature and input example. Please set `input_example` parameter when logging the model to auto infer the model signature.


RF RMSE: 4.941851373726247


### Experiment log edge model
Hieronder is te zien dat een linaire regressie model iets beter presteert, ook is deze sneller getraind. Daarom is het aangeraden om dit model te gebruiken als edge model. Een lineaire regressie model is licht, simpel en makkelijk te interpreteren.

In [0]:
edge_log = log[["run_id", "experiment_id","metrics.rmse","params.model_type","params.n_estimators"]]
edge_log = edge_log.dropna(subset=["params.model_type"])
display(edge_log)

run_id,experiment_id,metrics.rmse,params.model_type,params.n_estimators
75671a6e934f4a329399312610d7f667,541341117399548,4.941851373726247,RandomForest,50
8fecfb4e426d4cd7b3f2926fd8062652,541341117399548,3.2409702058105787,LinearRegression,null


# Deployment
Nu de data door de pipeline heen is en alle modellen getraind zijn, is het tijd voor de deployment. Om dit te doen zijn er run id's nodig die in de log staan, deze zijn te vinden in DataBricks onder "AI/ML" -> "Experiments". Om te weten welke het beste presteert kunnen we de bovenstaande dataframes raadplegen waar de benodigde informatie in staat.

### Cloud model deployment
Voor het cloud model wordt het het model gebruikt wat getraind is op de dataset waar lag features zijn toegevoegd en waar de maxDepth 5 is. Het model presteert bijna even goed als die van 10/15 alleen is een maxDepth van 5 sneller dan de hogere maxDepths. Door is experiments te kijken is dit ook terug te zien:
- maxDepth 5: 23 seconden
- maxDepth 10: 30 seconden
- maxDepth 15: 96 seconden\
Dit lijkt in eerste instantie niet veel maar wanneer er veel meer data binnen komt, schelen deze aantallen enorm. 

Er zou eventueel voor de maxDepth van 10 gekozen kunnen worden als de precisie van het model belangrijker is dan de tijd die gebruikt wordt voor het trainen.

In [0]:
# Model ophalen
model_uri = "runs:/ce70553bf9494add844e4050b417480a/model_depth_5"
# Model laden met bovenstaand variabel
loaded_model = mlflow.spark.load_model(model_uri)
#Model laten voorspellen
preds_cloud = loaded_model.transform(test)
#Voorspelling laten zien
display(preds_cloud.select("sales", "prediction"))

sales,prediction
26,15.567503441094313
13,20.982929253043597
18,18.441260889824353
15,19.227264834064794
24,19.227264834064794
22,21.920526364115027
32,19.973181249111125
21,21.231272473700777
15,22.42824834841183
18,21.920526364115027


### Edge model deployment
Voor het edge model zijn er twee modellen getraind en is er overduidelijk een beter, zowel in snelheid als in prestatie. Daarom wordt hieronder het linair regressiemodel aangeroepen op dezelfde manier zoals dat is gebeurt bij het cloud model.

In [0]:
# Model ophalen
model_uri = "runs:/8fecfb4e426d4cd7b3f2926fd8062652/edge_model_linear"
# Model laden
edge_model_loaded = mlflow.sklearn.load_model(model_uri)
# Voorspellingen maken
sample = X_test[:5]
actual = y_test[:5].ravel() # Naar 1D veranderen

preds_edge = edge_model_loaded.predict(sample)
preds_edge = preds_edge.ravel() # Naar 1D veranderen

compare_df = pd.DataFrame({
    "actual": actual,
    "predicted": preds_edge
})
compare_df

,actual,predicted
0,26,25.101218
1,40,33.239192
2,19,23.422859
3,27,27.544171
4,30,27.125734


# Model monitoring
Als laatste gaan we een belangrijk stuk toevoegen, namelijk het monitoren van de modellen en binnenstromende data. Het kan zo zijn dat de data veranderd en/of dat de modellen niet meer toereikend zijn. Het is belangrijk om zo snel mogelijk in te kunnen grijpen als dit gebeurt.

### Performance monitoring:

In [0]:
# Cloud monitoring
# (Nieuwe) predictions maken
preds_cloud_performance = loaded_model.transform(test)
# RMSE berekenen met evaluator van eerder
rmse = evaluator.evaluate(preds_cloud_performance)

print("Cloud RMSE:", rmse)

Cloud RMSE: 9.962707415865857


In [0]:
# Edge monitoring
# (Nieuwe) predictions maken
preds_edge_performance = edge_model_loaded.predict(X_test)
# RMSE berekenen
rmse = np.sqrt(mean_squared_error(y_test, preds_edge_performance))

print("Edge RMSE:", rmse)

Edge RMSE: 3.2409702058105787


### Drift detection:
In deze drift detection worden de gemiddeldes van de predictions en echte data vergeleken. Er is onderscheid gemaakt tussen de verschillende modellen omdat deze waardes gemiddeld ook verder uit elkaar liggen. Als dezelfde waardes gehanteerd zouden worden, zou er al sprake kunnen zijn van drift bij het edge model zonder dat dit wordt opgemerkt. Uiteraard kan de threshhold veranderd worden om eerder/later drift te detecteren.

Als er drift wordt gedetecteerd dan wordt er een nieuwe model getraind.

In [0]:
def retrain_cloud(train_df, assembler):
    from pyspark.ml.regression import RandomForestRegressor
    from pyspark.ml import Pipeline
    import mlflow

    rf = RandomForestRegressor(
        maxDepth=5,
        featuresCol="features",
        labelCol="sales"
    )

    pipeline = Pipeline(stages=[assembler, rf])

    model = pipeline.fit(train_df)

    print("Cloud model retrained")

    mlflow.spark.log_model(model, "cloud_model_retrained")

    return model

def retrain_edge(X_train, y_train):
    from sklearn.linear_model import LinearRegression
    import mlflow

    model = LinearRegression()
    model.fit(X_train, y_train)

    print("Edge model (Linear Regression) retrained")

    # log opnieuw in MLflow
    mlflow.sklearn.log_model(model, "edge_model_linear_retrained")

    return model

def detect_drift(preds, y_test, model=None):
    import numpy as np

    preds = np.array(preds)
    actual = np.array(y_test)

    avg_pred = np.mean(preds)
    avg_actual = np.mean(actual)

    print("Pred avg:", avg_pred)
    print("Actual avg:", avg_actual)

    if model == "edge":
        threshold = 5
    elif model == "cloud":
        threshold = 10
    else:
        raise ValueError("Model must be 'edge' or 'cloud'")

    if np.abs(avg_pred - avg_actual) > threshold:
        print("Drift detected, retraining")

        if model_type == "edge":
            return retrain_edge(X_train, y_train)

        elif model_type == "cloud":
            return retrain_cloud(train, assembler)

    else:
        print("No drift detected")



cdt = preds_cloud.select("prediction", "sales").toPandas()
cloud_drift = detect_drift(cdt["prediction"].values, cdt["sales"].values, "cloud")
edge_drift = detect_drift(preds_edge, y_test, "edge")


Pred avg: 52.326117795804144
Actual avg: 52.32850246876121
No drift detected
Pred avg: 30.096373690754877
Actual avg: 30.9125
No drift detected
